# Model Comparison: Time-Series Forecasting Baselines (Repo-Aligned)

This notebook compares the **current forecasting approach in this repo** (LightGBM + lag features + recursive multi-step forecasting) against alternative model families:

- LightGBM (current baseline)
- XGBoost
- Random Forest
- LSTM (deep learning)

The goal is an **apples-to-apples benchmark** aligned to the existing implementation:

- **Targets**: `current`, `temperature`, `z_rms`, `x_rms`, `z_peak`, `x_peak`, `noise`
- **Sampling**: 30-minute intervals
- **Lag window**: 48 steps (24 hours)
- **Forecast horizon**: 12 steps (6 hours)

This notebook is designed as a decision-support artifact: it produces comparable error curves per horizon and a practical recommendation for production use.


In [ ]:
import os
import sys
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt

# Make repo modules importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd()))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Import constants + helpers from the repo baseline
from backend.ml_engine import TARGETS, FREQ, LAG_STEPS, FORECAST_HORIZON, make_lag_features  # noqa

plt.rcParams["figure.figsize"] = (10, 4)


## Data loading

This benchmark can run from either:

1. **MySQL** using the repo function `backend.ml_engine.load_conveyor_data()` (requires DB env vars), or
2. A **CSV export** (recommended for repeatable experiments and CI-like runs).

If MySQL is not configured, the notebook falls back to CSV.


In [ ]:
from backend.ml_engine import load_conveyor_data  # noqa

DATA_CSV_PATH = os.environ.get("PM_DATA_CSV", "")  # optional


def load_data() -> pd.DataFrame:
    if DATA_CSV_PATH and os.path.exists(DATA_CSV_PATH):
        df = pd.read_csv(DATA_CSV_PATH)
        if "datetime" in df.columns:
            df["datetime"] = pd.to_datetime(df["datetime"])
            df = df.sort_values("datetime").set_index("datetime")
        else:
            raise ValueError("CSV must include a 'datetime' column")
        return df

    # Try MySQL via existing repo function
    df = load_conveyor_data()
    if df is None or df.empty:
        raise RuntimeError(
            "No data loaded from MySQL and PM_DATA_CSV not set. "
            "Set env var PM_DATA_CSV to a CSV export path, or configure DB env vars."
        )
    return df


df_raw = load_data()

# Keep only numeric target columns for forecasting benchmarks
missing = [c for c in TARGETS if c not in df_raw.columns]
if missing:
    raise ValueError(f"Loaded data is missing required target columns: {missing}")

df = df_raw.copy()
df.index = pd.to_datetime(df.index)
df = df.sort_index()

df[TARGETS].tail(3)


## Benchmark protocol (repo-aligned)

We evaluate models under the same conditions as your production forecaster:

- **Lag features**: 48-step history (`LAG_STEPS`) for all targets
- **Multi-target**: one regressor per target (tree models) vs joint model (LSTM)
- **Forecasting mode**:
  - Tree baselines use the **same recursive roll-forward** as the current LightGBM implementation.
  - LSTM uses a **direct multi-step** mapping (predict 12 future steps at once) to reduce error drift; we still report the same horizon metrics.

### Time split
Because this is time series, we use a chronological split.

- Train: earliest data
- Test: last `test_days` days (default 7)

### Metrics
For each target and each horizon step (1..12), we report:
- RMSE
- MAE

We also compute an aggregate score across targets (mean RMSE after per-target normalization by the target’s training-set std).


In [ ]:
@dataclass
class SplitData:
    train_df: pd.DataFrame
    test_df: pd.DataFrame


def time_split(df: pd.DataFrame, test_days: int = 7) -> SplitData:
    # With 30min frequency, 1 day = 48 steps
    test_steps = test_days * 48
    if len(df) <= test_steps + LAG_STEPS + FORECAST_HORIZON:
        raise ValueError("Not enough data for split + lags + horizon")

    train_df = df.iloc[: -test_steps].copy()
    test_df = df.iloc[-test_steps:].copy()
    return SplitData(train_df=train_df, test_df=test_df)


def build_supervised(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build X (lag features) and y (current step targets) exactly like ml_engine.run_pipeline."""
    df_numeric = df[TARGETS].copy()
    data = make_lag_features(df_numeric, LAG_STEPS)
    X = data.drop(columns=TARGETS)
    y = data[TARGETS]
    return X, y


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def horizon_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


In [ ]:
def recursive_forecast_one_origin(
    models_by_target: Dict[str, object],
    history_df: pd.DataFrame,
    X_cols: List[str],
    horizon: int = FORECAST_HORIZON,
) -> pd.DataFrame:
    """Repo-aligned recursive loop: predict 1 step, append preds, repeat."""
    buffer = history_df.iloc[-LAG_STEPS:].copy()
    forecast_dict = {tgt: [] for tgt in TARGETS}

    # Determine base columns from lag feature names
    base_cols = [c.split("_lag")[0] for c in X_cols if "_lag" in c]
    base_cols = list(set(base_cols))

    for _ in range(horizon):
        input_row = {}
        for col in base_cols:
            if col in buffer.columns:
                for lag in range(1, LAG_STEPS + 1):
                    lag_col = f"{col}_lag{lag}"
                    if lag_col in X_cols:
                        input_row[lag_col] = buffer.iloc[-lag][col]

        X_pred = pd.DataFrame([input_row]).reindex(columns=X_cols).fillna(0)

        preds_step = {}
        for tgt in TARGETS:
            model = models_by_target[tgt]
            val = float(model.predict(X_pred)[0])
            preds_step[tgt] = val
            forecast_dict[tgt].append(val)

        new_row = buffer.iloc[-1].copy()
        for tgt in TARGETS:
            new_row[tgt] = preds_step[tgt]

        buffer = pd.concat([buffer, new_row.to_frame().T]).iloc[-LAG_STEPS:]

    # Index is not essential for metric computation
    return pd.DataFrame(forecast_dict)


def evaluate_recursive_model(
    models_by_target: Dict[str, object],
    full_df: pd.DataFrame,
    split: SplitData,
    origins: Optional[List[int]] = None,
    horizon: int = FORECAST_HORIZON,
) -> Dict[str, object]:
    """Evaluate on multiple forecast origins inside test window."""
    X_train, y_train = build_supervised(split.train_df)
    X_cols = list(X_train.columns)

    test_df = split.test_df
    if origins is None:
        # Choose ~20 evenly spaced origins, ensuring enough future points
        max_origin = len(test_df) - horizon - 1
        origins = np.linspace(LAG_STEPS, max_origin, num=20, dtype=int).tolist()

    # Precompute normalization (train std) for aggregate scoring
    train_std = split.train_df[TARGETS].std().replace(0, np.nan)

    per_target_per_h: Dict[str, List[Dict[str, float]]] = {t: [] for t in TARGETS}

    for origin in origins:
        # Build the history window as: train + test up to origin
        hist = pd.concat([split.train_df, test_df.iloc[:origin]]).copy()
        if len(hist) < LAG_STEPS:
            continue

        preds = recursive_forecast_one_origin(models_by_target, hist[TARGETS], X_cols, horizon=horizon)
        true_future = test_df[TARGETS].iloc[origin : origin + horizon].reset_index(drop=True)

        # per-horizon step metrics
        for h in range(horizon):
            for tgt in TARGETS:
                m = horizon_metrics(
                    np.array([true_future.loc[h, tgt]]),
                    np.array([preds.loc[h, tgt]]),
                )
                per_target_per_h[tgt].append({"h": h + 1, **m})

    # Aggregate: mean RMSE per horizon, normalized by train std
    agg_by_h = []
    for h in range(1, horizon + 1):
        rmses_norm = []
        for tgt in TARGETS:
            vals = [x["rmse"] for x in per_target_per_h[tgt] if x["h"] == h]
            if not vals:
                continue
            denom = float(train_std.get(tgt, np.nan))
            if not math.isfinite(denom) or denom == 0:
                continue
            rmses_norm.append(float(np.mean(vals)) / denom)
        agg_by_h.append({"h": h, "mean_norm_rmse": float(np.mean(rmses_norm)) if rmses_norm else np.nan})

    return {
        "per_target_per_h": per_target_per_h,
        "agg_by_h": pd.DataFrame(agg_by_h),
        "origins": origins,
    }


## Baseline 1: LightGBM (current repo approach)

Your production pipeline trains **one LightGBM regressor per target** using lag features, then performs a **recursive forecast**.

Here we replicate that behavior for benchmarking.


In [ ]:
import lightgbm as lgb


def train_lightgbm_per_target(X_train: pd.DataFrame, y_train: pd.DataFrame) -> Dict[str, object]:
    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "verbosity": -1,
        "seed": 42,
    }

    models = {}
    for tgt in TARGETS:
        ds = lgb.Dataset(X_train, y_train[tgt])
        model = lgb.train(params, ds, num_boost_round=300)
        models[tgt] = model
    return models


split = time_split(df, test_days=7)
X_train, y_train = build_supervised(split.train_df)

lgbm_models = train_lightgbm_per_target(X_train, y_train)

lgbm_results = evaluate_recursive_model(lgbm_models, df, split)

lgbm_results["agg_by_h"].head()


In [ ]:
def plot_aggregate(agg_df: pd.DataFrame, title: str):
    d = agg_df.copy()
    plt.plot(d["h"], d["mean_norm_rmse"], marker="o")
    plt.xticks(range(1, FORECAST_HORIZON + 1))
    plt.grid(True, alpha=0.3)
    plt.title(title)
    plt.xlabel("Horizon step (1..12)")
    plt.ylabel("Mean normalized RMSE (lower is better)")
    plt.show()


plot_aggregate(lgbm_results["agg_by_h"], "LightGBM baseline: aggregate normalized RMSE by horizon")


In [ ]:
def per_target_summary(per_target_per_h: Dict[str, List[Dict[str, float]]]) -> pd.DataFrame:
    rows = []
    for tgt, items in per_target_per_h.items():
        dfm = pd.DataFrame(items)
        if dfm.empty:
            continue
        # Mean by horizon
        g = dfm.groupby("h").agg(rmse=("rmse", "mean"), mae=("mae", "mean")).reset_index()
        g["target"] = tgt
        rows.append(g)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


lgbm_per_target = per_target_summary(lgbm_results["per_target_per_h"])
lgbm_per_target.head()


In [ ]:
def plot_target_curves(summary_df: pd.DataFrame, model_name: str, metric: str = "rmse"):
    for tgt in TARGETS:
        d = summary_df[summary_df["target"] == tgt]
        if d.empty:
            continue
        plt.plot(d["h"], d[metric], marker="o", label=tgt)

    plt.xticks(range(1, FORECAST_HORIZON + 1))
    plt.grid(True, alpha=0.3)
    plt.title(f"{model_name}: {metric.upper()} by horizon (per target)")
    plt.xlabel("Horizon step (1..12)")
    plt.ylabel(metric.upper())
    plt.legend(ncol=2, fontsize=8)
    plt.show()


plot_target_curves(lgbm_per_target, "LightGBM", metric="rmse")


## Baseline 2: Random Forest (sklearn)

Random Forest is a strong non-linear baseline that is usually **more stable but less accurate** than boosting on many tabular problems. It also tends to be heavier at inference if the forest is large.

We keep the setup identical: one regressor per target, lag features, recursive forecast evaluation.


In [ ]:
def train_random_forest_per_target(X_train: pd.DataFrame, y_train: pd.DataFrame) -> Dict[str, object]:
    models = {}
    for tgt in TARGETS:
        model = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        )
        model.fit(X_train, y_train[tgt])
        models[tgt] = model
    return models


rf_models = train_random_forest_per_target(X_train, y_train)
rf_results = evaluate_recursive_model(rf_models, df, split)
plot_aggregate(rf_results["agg_by_h"], "Random Forest: aggregate normalized RMSE by horizon")


## Baseline 3: XGBoost

XGBoost is typically competitive with LightGBM for tabular time-series features.

If `xgboost` is not installed, this section can be skipped.


In [ ]:
def try_import_xgboost():
    try:
        import xgboost as xgb  # type: ignore
        return xgb
    except Exception as e:
        print("[SKIP] xgboost not available:", e)
        return None


xgb = try_import_xgboost()

xgb_results = None
if xgb is not None:
    def train_xgboost_per_target(X_train: pd.DataFrame, y_train: pd.DataFrame) -> Dict[str, object]:
        models = {}
        for tgt in TARGETS:
            model = xgb.XGBRegressor(
                n_estimators=600,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="reg:squarederror",
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X_train, y_train[tgt])
            models[tgt] = model
        return models

    xgb_models = train_xgboost_per_target(X_train, y_train)
    xgb_results = evaluate_recursive_model(xgb_models, df, split)
    plot_aggregate(xgb_results["agg_by_h"], "XGBoost: aggregate normalized RMSE by horizon")


## Comparison: aggregate curves

This plot overlays aggregate normalized RMSE across horizons for all available models.


In [ ]:
def overlay_aggregate(curves: Dict[str, pd.DataFrame]):
    for name, dfc in curves.items():
        d = dfc.copy()
        plt.plot(d["h"], d["mean_norm_rmse"], marker="o", label=name)

    plt.xticks(range(1, FORECAST_HORIZON + 1))
    plt.grid(True, alpha=0.3)
    plt.title("Aggregate normalized RMSE by horizon (lower is better)")
    plt.xlabel("Horizon step (1..12)")
    plt.ylabel("Mean normalized RMSE")
    plt.legend()
    plt.show()


curves = {
    "LightGBM": lgbm_results["agg_by_h"],
    "RandomForest": rf_results["agg_by_h"],
}
if xgb_results is not None:
    curves["XGBoost"] = xgb_results["agg_by_h"]

overlay_aggregate(curves)


## Baseline 4: LSTM (direct multi-step)

For the LSTM baseline we use a **direct multi-step** approach:

- Input: last 48 timesteps of all 7 targets (shape: `48 x 7`)
- Output: next 12 timesteps of all 7 targets (shape: `12 x 7`)

This avoids recursive error accumulation in neural nets and is a common approach for multi-horizon forecasting.

If `tensorflow` is not installed, this section will be skipped.


In [ ]:
def try_import_tf():
    try:
        import tensorflow as tf  # type: ignore
        return tf
    except Exception as e:
        print("[SKIP] tensorflow not available:", e)
        return None


tf = try_import_tf()


In [ ]:
def make_sequences_multi_step(
    df_targets: pd.DataFrame,
    lookback: int = LAG_STEPS,
    horizon: int = FORECAST_HORIZON,
) -> Tuple[np.ndarray, np.ndarray]:
    values = df_targets[TARGETS].values.astype(np.float32)
    Xs, Ys = [], []
    for i in range(lookback, len(values) - horizon):
        Xs.append(values[i - lookback : i])
        Ys.append(values[i : i + horizon])
    return np.stack(Xs), np.stack(Ys)


def evaluate_direct_multistep(
    y_true: np.ndarray,  # (N, H, D)
    y_pred: np.ndarray,  # (N, H, D)
    train_std: pd.Series,
) -> Dict[str, object]:
    per_target_per_h: Dict[str, List[Dict[str, float]]] = {t: [] for t in TARGETS}

    H = y_true.shape[1]
    for h in range(H):
        for j, tgt in enumerate(TARGETS):
            mt = horizon_metrics(y_true[:, h, j], y_pred[:, h, j])
            per_target_per_h[tgt].append({"h": h + 1, **mt})

    agg_by_h = []
    for h in range(1, H + 1):
        rmses_norm = []
        for tgt in TARGETS:
            denom = float(train_std.get(tgt, np.nan))
            if not math.isfinite(denom) or denom == 0:
                continue
            vals = [x["rmse"] for x in per_target_per_h[tgt] if x["h"] == h]
            if vals:
                rmses_norm.append(float(np.mean(vals)) / denom)
        agg_by_h.append({"h": h, "mean_norm_rmse": float(np.mean(rmses_norm)) if rmses_norm else np.nan})

    return {
        "per_target_per_h": per_target_per_h,
        "agg_by_h": pd.DataFrame(agg_by_h),
    }


In [ ]:
lstm_results = None

if tf is not None:
    # Build sequences from train/test split
    train_targets = split.train_df[TARGETS].copy()
    test_targets = split.test_df[TARGETS].copy()

    Xtr, Ytr = make_sequences_multi_step(train_targets)
    Xte, Yte = make_sequences_multi_step(test_targets)

    # Normalize using train stats (per target)
    mu = train_targets.mean().astype(np.float32)
    sig = train_targets.std().replace(0, np.nan).astype(np.float32)

    def norm(x):
        return (x - mu.values) / sig.values

    Xtr_n = norm(Xtr)
    Ytr_n = norm(Ytr)
    Xte_n = norm(Xte)

    # Simple LSTM
    D = len(TARGETS)
    H = FORECAST_HORIZON

    inp = tf.keras.Input(shape=(LAG_STEPS, D))
    x = tf.keras.layers.LSTM(64, return_sequences=False)(inp)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dense(H * D)(x)
    out = tf.keras.layers.Reshape((H, D))(x)

    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

    # Train briefly (tune as needed)
    history = model.fit(
        Xtr_n,
        Ytr_n,
        validation_split=0.1,
        epochs=10,
        batch_size=128,
        verbose=1,
    )

    # Predict and unnormalize
    Yhat_n = model.predict(Xte_n)
    Yhat = (Yhat_n * sig.values) + mu.values

    lstm_results = evaluate_direct_multistep(Yte, Yhat, train_std=split.train_df[TARGETS].std().replace(0, np.nan))
    plot_aggregate(lstm_results["agg_by_h"], "LSTM (direct multi-step): aggregate normalized RMSE by horizon")


## Comparison: include LSTM if available


In [ ]:
curves2 = dict(curves)
if lstm_results is not None:
    curves2["LSTM_direct"] = lstm_results["agg_by_h"]

overlay_aggregate(curves2)


## Decision guide (tailored to this repo)

Use the plots above to make a production decision. Key points for your setup:

### 1) If you want the simplest and strongest baseline
- **Choose LightGBM** (current approach) when it beats or ties others on aggregate curve.
- Pros: strong tabular performance with lag features, fast training/inference, easy deployment.
- Cons: recursive forecasts can drift at longer horizons; you may need periodic retraining.

### 2) If you need a safety-first baseline
- **Choose Random Forest** if it is slightly worse but notably more stable (less spiky) across horizons.
- Pros: robust to some noise/outliers.
- Cons: often less accurate than boosting; can be slower at inference.

### 3) If you want best tabular accuracy and can add a dependency
- **Choose XGBoost** if it consistently improves horizons vs LightGBM.
- Pros: competitive accuracy.
- Cons: extra dependency; tuning can be time-consuming.

### 4) If you have enough data and want multi-step stability
- **Choose LSTM (direct multi-step)** only if it clearly wins on longer horizons and you can support the ops cost.
- Pros: direct multi-horizon output reduces recursive error accumulation.
- Cons: more complex training, harder debugging, usually needs more data, dependency footprint.

### Practical recommendation for your current architecture
- You already retrain on a schedule and serve forecasts every 5 minutes.
- In most predictive maintenance dashboards, a **reliable and interpretable model** wins over a marginal RMSE gain.
- If LightGBM and XGBoost are close, prefer **LightGBM** to keep ops simple.

### Next steps (if you want to improve the current LightGBM approach)
- Evaluate **direct multi-step** tree models (train 12 separate models per target, one per horizon) to reduce recursion drift.
- Add **exogenous features** (time-of-day, day-of-week, maintenance events) if available.
- Track **data delay** and **imputation flags** as model features (you already generate `*_error_flag` columns during imputation).


In [ ]:
# Optional: quick leaderboard snapshot (mean over horizons)

def mean_over_horizons(agg_df: pd.DataFrame) -> float:
    d = agg_df.copy()
    return float(np.nanmean(d["mean_norm_rmse"]))

leaderboard = []
for name, d in curves2.items():
    leaderboard.append({"model": name, "mean_norm_rmse": mean_over_horizons(d)})

pd.DataFrame(leaderboard).sort_values("mean_norm_rmse")
